In [1]:
import os
from pathlib import Path
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.documents import Document
from langchain_groq import ChatGroq
from langchain_classic.retrievers import ContextualCompressionRetriever
from langchain_classic.retrievers.document_compressors import (
    LLMChainExtractor
)


In [2]:
# Google API key environment variable থেকে নিচ্ছি
groq_api_key = os.getenv("GROQ_API_KEY")


# local embedding model import
MODEL_PATH = Path.home() / "rag-learning-practice" / "models" / "all-MiniLM-L6-v2"

# Local embedding model setup
embeddings = HuggingFaceEmbeddings(
    model_name=str(MODEL_PATH),
    model_kwargs={
        "device": "cpu",
        "local_files_only": True
    },
    encode_kwargs={
        "normalize_embeddings": True
    }
)


/home/mehedi-hassan/rag-learning-practice/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 10064.38it/s]


In [3]:
documents = [
    Document(
        page_content=(
            "LangChain is a framework for building LLM applications. "
            "It provides tools for prompts, chains, agents, and retrieval. "
            "Retrieval helps provide external knowledge to the model."
        )
    ),
    Document(
        page_content=(
            "Chroma is a vector database. "
            "It stores embeddings and supports similarity search. "
            "It can persist data locally."
        )
    ),
]

vector_store = Chroma(
    collection_name="contextual_compression_doc",
    embedding_function=embeddings,
    persist_directory="../all_vector_stores/chroma_contextual_compression"
)

vector_store.add_documents(documents)

['f09a864e-ca24-43ed-861e-1747dd989e09',
 'bdb6fae8-4925-482f-aab4-34692e91948a']

In [4]:
base_retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 2}
)

llm = ChatGroq(
    model="openai/gpt-oss-20b",
    groq_api_key=groq_api_key,
    temperature=0
)

# LLMChainExtractor প্রতিটি document থেকে query-র জন্য relevant অংশ extract করে
compressor = LLMChainExtractor.from_llm(llm)

In [5]:
# Base retriever + compressor = ContextualCompressionRetriever
compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor,
    base_retriever=base_retriever
)

query = "What is Chroma used for?"

results = compression_retriever.invoke(query)

print("Query:", query)
print("-" * 50)

for i, doc in enumerate(results, start=1):
    print(f"{i}. {doc.page_content}")

Query: What is Chroma used for?
--------------------------------------------------
1. Chroma is a vector database. It stores embeddings and supports similarity search. It can persist data locally.
